In [67]:
def pagerank(G, alpha=0.85, personalization=None, max_iter=100, tol=1.0e-6, nstart=None, weight='weight', dangling=None):
    if len(G) == 0:
        return {}

    if not G.is_directed():
        D = G.to_directed()
    else:
        D = G

    # Create a copy in (right) stochastic form
    W = nx.stochastic_graph(D, weight=weight)
    N = W.number_of_nodes()

    # Choose fixed starting vector if not given
    if nstart is None:
        x = dict.fromkeys(W, 1.0 / N)
    else:
        # Normalized nstart vector
        s = float(sum(nstart.values()))
        x = dict((k, v / s) for k, v in nstart.items())

    if personalization is None:
        # Assign uniform personalization vector if not given
        p = dict.fromkeys(W, 1.0 / N)
    else:
        missing = set(G) - set(personalization)
        if missing:
            raise NetworkXError('Personalization dictionary must have a value for every node. Missing nodes %s' % missing)
        s = float(sum(personalization.values()))
        p = dict((k, v / s) for k, v in personalization.items())

    if dangling is None:
        # Use personalization vector if dangling vector not specified
        dangling_weights = p
    else:
        missing = set(G) - set(dangling)
        if missing:
            raise NetworkXError('Dangling node dictionary must have a value for every node. Missing nodes %s' % missing)
        s = float(sum(dangling.values()))
        dangling_weights = dict((k, v/s) for k, v in dangling.items())

    dangling_nodes = [n for n in W if W.out_degree(n, weight=weight) == 0.0]

     # power iteration: make up to max_iter iterations
    for _ in range(max_iter):
        xlast = x
        x = dict.fromkeys(xlast.keys(), 0)
        danglesum = alpha * sum(xlast[n] for n in dangling_nodes)
        for n in x:
            # this matrix multiply looks odd because it is
            # doing a left multiply x^T=xlast^T*W
            for nbr in W[n]:
                x[nbr] += alpha * xlast[n] * W[n][nbr][weight]
            x[n] += danglesum * dangling_weights[n] + (1.0 - alpha) * p[n]

        # check convergence, l1 norm
        err = sum([abs(x[n] - xlast[n]) for n in x])
        if err < N*tol:
            return x
    raise NetworkXError('Pagerank: power iteration failed to converge in %d iterations.' % max_iter)

In [68]:
import networkx as nx

In [69]:
G = nx.barabasi_albert_graph(60, 41)
pr = nx.pagerank(G, 0.4)

In [70]:
print(pr)

{0: 0.028045469081308984, 1: 0.01316576575288953, 2: 0.013170395884866681, 3: 0.013365873511066093, 4: 0.01278260480400472, 5: 0.012975421438596355, 6: 0.012955115980009951, 7: 0.01298159718471058, 8: 0.013173495962644255, 9: 0.012164075812757998, 10: 0.013170042860269144, 11: 0.012959443013775173, 12: 0.013170778704311577, 13: 0.013183297409465842, 14: 0.012578940710442724, 15: 0.012963522542363598, 16: 0.012180361579157722, 17: 0.012367742705001935, 18: 0.012958514832693216, 19: 0.013375299171964873, 20: 0.013161117680469016, 21: 0.012963377991378488, 22: 0.01357569343594802, 23: 0.01296419739153601, 24: 0.013776874764438165, 25: 0.013170940829407033, 26: 0.013372170937309382, 27: 0.012756225776979985, 28: 0.012967539290986837, 29: 0.013576809298736047, 30: 0.012373388891042338, 31: 0.013581667756266529, 32: 0.012353762403116614, 33: 0.012753895172974324, 34: 0.012959599245501106, 35: 0.012560678227920433, 36: 0.012972599408506818, 37: 0.013174305919834976, 38: 0.012769198718249424, 

In [71]:
# Import required library
import networkx as nx

# Create a random graph (Barabási–Albert model)
G = nx.barabasi_albert_graph(60, 41)

# Compute PageRank using NetworkX built-in function
pr = nx.pagerank(G, alpha=0.4)

# Display PageRank values
print("PageRank values for each node:")
print(pr)

PageRank values for each node:
{0: 0.028095991969604855, 1: 0.012961614969290276, 2: 0.012534762508007426, 3: 0.012963413208109351, 4: 0.012353945321260627, 5: 0.013159018696080778, 6: 0.012748195460114409, 7: 0.013575839494054099, 8: 0.011983755324975148, 9: 0.012554963262663985, 10: 0.012748127477231756, 11: 0.0133560545383306, 12: 0.013376060502066262, 13: 0.013182801294474576, 14: 0.01315666546911845, 15: 0.01295524082778994, 16: 0.012787671457776829, 17: 0.012953566636910116, 18: 0.013153969111922898, 19: 0.012570768214481335, 20: 0.01338546147295449, 21: 0.012778357852491483, 22: 0.012363989299203673, 23: 0.012557529187428095, 24: 0.011979293821535859, 25: 0.012975882266611933, 26: 0.012964616081839494, 27: 0.013157127289836519, 28: 0.012956116529469021, 29: 0.012589731033464797, 30: 0.012766827428057969, 31: 0.013365492875862896, 32: 0.013575181007739454, 33: 0.012977539319934414, 34: 0.013777266434147156, 35: 0.013162074348581533, 36: 0.012170754663558409, 37: 0.012970766580779